In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.basemap import Basemap
import numpy as np
from torchvision.utils import save_image
from torchmetrics.regression import PearsonCorrCoef
from eofs.standard import Eof
from torch import nn

In [ ]:
class Encoder(nn.Module):
    def __init__(self, in_channels=1, latent_dim=256, input_height=None, input_width=None):
        super().__init__()

        assert input_height is not None and input_width is not None

        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, 64, 4, 2, 1),
            nn.GroupNorm(8, 64),
            nn.ReLU(inplace=True),

            nn.Conv2d(64, 128, 4, 2, 1),
            nn.GroupNorm(8, 128),
            nn.ReLU(inplace=True),

            nn.Conv2d(128, 256, 4, 2, 1),
            nn.GroupNorm(8, 256),
            nn.ReLU(inplace=True),
        )


        with torch.no_grad():
            dummy = torch.zeros(1, in_channels, input_height, input_width)
            out = self.conv(dummy)
            self.final_shape = out.shape[1:]
            flattened_size = out.numel()

        self.fc_mu = nn.Linear(flattened_size, latent_dim)
        self.fc_logvar = nn.Linear(flattened_size, latent_dim)

    def forward(self, x):
        x = self.conv(x)
        x = x.flatten(1)
        mu = self.fc_mu(x)
        logvar = self.fc_logvar(x)
        return mu, logvar


In [ ]:
def reparameterize(mu, logvar):
    std = torch.exp(0.5 * logvar)
    eps = torch.randn_like(std)
    return mu + eps * std

In [ ]:
class Decoder(nn.Module):
    def __init__(self, latent_dim=256, out_channels=1, final_shape=None):
        super().__init__()

        assert final_shape is not None


        self.final_shape = final_shape
        C, H, W = final_shape

        self.fc = nn.Linear(latent_dim, C * H * W)

        self.deconv = nn.Sequential(
            nn.ConvTranspose2d(256, 128, 4, 2, 1),
            nn.GroupNorm(8, 128),
            nn.ReLU(inplace=True),

            nn.ConvTranspose2d(128, 64, 4, 2, 1),
            nn.GroupNorm(8, 64),
            nn.ReLU(inplace=True),

            nn.ConvTranspose2d(64, out_channels, 4, 2, 1),
        )

    def forward(self, z):
        x = self.fc(z)
        x = x.view(z.size(0), *self.final_shape)
        return self.deconv(x)

In [ ]:
class VanillaVAE(nn.Module):
    def __init__(self, in_channels=1, latent_dim=256, input_height=None, input_width=None):
        super().__init__()

        self.encoder = Encoder(
            in_channels=in_channels,
            latent_dim=latent_dim,
            input_height=input_height,
            input_width=input_width
        )

        self.decoder = Decoder(
            latent_dim=latent_dim,
            out_channels=in_channels,
            final_shape=self.encoder.final_shape
        )

    def forward(self, x):
        mu, logvar = self.encoder(x)
        z = reparameterize(mu, logvar)
        recon = self.decoder(z)
        return recon, mu, logvar

In [ ]:
import torch.nn.functional as F
def vae_loss(recon, x, mu, logvar, beta=1.0):
    recon_loss = F.l1_loss(
        recon, x, reduction="none"
    ).sum(dim=(1,2,3)).mean()

    kl_loss = -0.5 * torch.sum(
        1 + logvar - mu.pow(2) - logvar.exp(),
        dim=1
    ).mean()

    return recon_loss + beta * kl_loss, recon_loss, kl_loss